# Text Feature Extraction — IEMOCAP

Encoder: configurable via `MODEL_NAME` in the config cell (default: `roberta-large`)  
Feature: CLS token from last hidden state → shape `(HIDDEN_SIZE,)` per utterance  
Output: `Dataset/Processed/IEMOCAP/features/text_{MODEL_TAG}.pt`  
Format: `dict {utt_id: np.array(HIDDEN_SIZE,)}`

In [1]:
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from transformers import AutoTokenizer, AutoModel
from tqdm.notebook import tqdm

In [ ]:
PROJECT_ROOT = Path("/mnt/Work/ML/Thesis/BMVC/Hopeful")
DATA_ROOT    = PROJECT_ROOT / "Dataset/Processed/IEMOCAP"
FEATURES_DIR = DATA_ROOT / "features"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 32
MAX_LEN    = 512

# ── Swap model here ───────────────────────────────────────────────────────────
MODEL_NAME = "roberta-large"          # any HuggingFace AutoModel-compatible ID
# Examples:
#   "roberta-large"                      → text_roberta_large.pt        (hidden=1024)
#   "bert-base-uncased"                  → text_bert_base_uncased.pt    (hidden=768)
#   "bert-large-uncased"                 → text_bert_large_uncased.pt   (hidden=1024)
#   "microsoft/deberta-v3-large"         → text_microsoft_deberta_v3_large.pt (hidden=1024)
#   "google/electra-large-discriminator" → text_google_electra_large_discriminator.pt
# ─────────────────────────────────────────────────────────────────────────────
MODEL_TAG  = MODEL_NAME.replace("/", "_").replace("-", "_")
print(f"Model    : {MODEL_NAME}")
print(f"Tag      : {MODEL_TAG}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device   : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM free: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")

In [3]:
labels = pd.read_csv(DATA_ROOT / "labels.csv")
print(f"Utterances : {len(labels)}")
print(f"Columns    : {labels.columns.tolist()}")
labels.head(3)

Utterances : 10039
Columns    : ['utt_id', 'session', 'dialog', 'speaker', 'emotion', 'valence', 'arousal', 'dominance', 'start_time', 'end_time', 'transcription', 'audio_path', 'video_path', 'text_path']


,utt_id,session,dialog,speaker,emotion,valence,arousal,dominance,start_time,end_time,transcription,audio_path,video_path,text_path
0,Ses01F_impro01_F000,Session1,Ses01F_impro01,F,neu,2.5,2.5,2.5,6.2901,8.2357,Excuse me.,audio/Ses01F_impro01_F000.wav,video/Ses01F_impro01_F000.mp4,text/Ses01F_impro01_F000.txt
1,Ses01F_impro01_F001,Session1,Ses01F_impro01,F,neu,2.5,2.5,2.5,10.0100,11.3925,Yeah.,audio/Ses01F_impro01_F001.wav,video/Ses01F_impro01_F001.mp4,text/Ses01F_impro01_F001.txt
2,Ses01F_impro01_F002,Session1,Ses01F_impro01,F,neu,2.5,2.5,2.5,14.8872,18.0175,Is there a problem?,audio/Ses01F_impro01_F002.wav,video/Ses01F_impro01_F002.mp4,text/Ses01F_impro01_F002.txt


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModel.from_pretrained(MODEL_NAME)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)
model = model.to(DEVICE)
HIDDEN_SIZE = model.config.hidden_size
print(f"Hidden size : {HIDDEN_SIZE}")
print(f"Params      : {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Hidden size : 1024
Params      : 355M


In [5]:
def extract_batch(texts: list[str]) -> np.ndarray:
    """Returns CLS-token embeddings, shape (B, HIDDEN_SIZE)."""
    enc = tokenizer(
        texts,
        max_length=MAX_LEN,
        truncation=True,
        padding=True,
        return_tensors="pt",
    )
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    with torch.no_grad():
        out = model(**enc)
    return out.last_hidden_state[:, 0, :].cpu().numpy()

In [ ]:
features: dict[str, np.ndarray] = {}

utt_ids    = labels["utt_id"].tolist()
text_paths = labels["text_path"].tolist()

batch_ids: list[str]  = []
batch_texts: list[str] = []

for i, (uid, tpath) in enumerate(tqdm(zip(utt_ids, text_paths), total=len(utt_ids), desc="Extracting")):
    text = (DATA_ROOT / tpath).read_text(encoding="utf-8").strip()  # paths relative to DATA_ROOT
    batch_ids.append(uid)
    batch_texts.append(text if text else "[empty]")

    if len(batch_texts) == BATCH_SIZE or i == len(utt_ids) - 1:
        feats = extract_batch(batch_texts)
        for bid, feat in zip(batch_ids, feats):
            features[bid] = feat
        batch_ids.clear()
        batch_texts.clear()

print(f"Extracted : {len(features)} utterances")

In [7]:
out_path = FEATURES_DIR / f"text_{MODEL_TAG}.pt"
torch.save(features, out_path)
size_mb = out_path.stat().st_size / 1e6
print(f"Saved  : {out_path}")
print(f"Size   : {size_mb:.1f} MB")

Saved  : Dataset/Processed/IEMOCAP/features/text_roberta_large.pt
Size   : 62.8 MB


In [8]:
loaded = torch.load(out_path, weights_only=False)

assert len(loaded) == 10039, f"Expected 10039, got {len(loaded)}"

sample_key  = next(iter(loaded))
sample_feat = loaded[sample_key]
assert sample_feat.shape == (1024,), f"Expected (1024,), got {sample_feat.shape}"

all_feats = np.stack(list(loaded.values()))
print(f"Count   : {len(loaded)}")
print(f"Shape   : {sample_feat.shape}")
print(f"Mean    : {all_feats.mean():.4f}")
print(f"Std     : {all_feats.std():.4f}")
print(f"Min     : {all_feats.min():.4f}")
print(f"Max     : {all_feats.max():.4f}")
print(f"Has NaN : {np.isnan(all_feats).any()}")
print(f"Has Inf : {np.isinf(all_feats).any()}")

sample_row  = labels[labels.utt_id == sample_key].iloc[0]
sample_text = (DATA_ROOT / sample_row.text_path).read_text(encoding="utf-8").strip()
l2_norm     = np.linalg.norm(sample_feat)
print(f"\nSample utt_id : {sample_key}")
print(f"Emotion       : {sample_row.emotion}")
print(f"Text          : {sample_text[:100]}")
print(f"L2 norm       : {l2_norm:.2f}")

Count   : 10039
Shape   : (1024,)
Mean    : -0.0315
Std     : 0.9889
Min     : -31.6279
Max     : 3.4486
Has NaN : False
Has Inf : False

Sample utt_id : Ses01F_impro01_F000
Emotion       : neu
Text          : Excuse me.
L2 norm       : 31.66
